# Tutorial 4 — Mixed-type landscape (categorical + ordinal)

**Dataset:** Eisenberg *et al.* 2025 (JACS) — electrochemical reduction of benzaldehyde.

Three design variables of two different kinds:

| variable      | type        | levels |
| ---           | ---         | ---    |
| `electrode`   | categorical | {Cu, Pb, Bi, graphite} |
| `pH`          | ordinal     | {2, 7, 13}  |
| `potential_V` | ordinal     | {−0.6, −0.8, −1.0} V |

Full factorial = **36 runs**. `selectivity_scalar ∈ [−100, 100]` is the paper's signed dimer-vs-alcohol selectivity — high = dimer-selective; low = alcohol-selective. We maximize.

This is the common case where you have **one nominal axis** (which chemical / which device) and **one or more ordered axes** (intensive variables on a grid).

In [ ]:
import pandas as pd
from graphfla.landscape import Landscape
from graphfla.analysis import (
    lo_ratio, autocorrelation, fitness_distance_corr,
    classify_epistasis, single_mutation_effects,
)

df = pd.read_csv('../data/Chemistry/EChem/aldehyde_electro_36.csv')
df[['electrode', 'pH', 'potential_V', 'selectivity_scalar']].head()

## 1. Build the landscape with per-axis types

In [ ]:
X = df[['electrode', 'pH', 'potential_V']]
f = df['selectivity_scalar']

ls = Landscape(maximize=True)
ls.build_from_data(
    X, f,
    data_types={
        'electrode':   'categorical',
        'pH':          'ordinal',
        'potential_V': 'ordinal',
    },
    verbose=False,
)
print(f'nodes: {ls.graph.vcount()}, edges: {ls.graph.ecount()}')
# Per node: (4-1) electrode + (3-1) pH + (3-1) V = 7 neighbours; 36 * 7 / 2 = 126 undirected edges.

## 2. Topographical features

In [ ]:
print(f'lo_ratio        : {lo_ratio(ls):.4f}')
print(f'autocorr (lag1) : {autocorrelation(ls):.4f}')
print(f'fdc (Spearman)  : {fitness_distance_corr(ls):.4f}')
print('\nEpistasis (4-node squares):')
for k, v in classify_epistasis(ls).items():
    print(f'  {k:>26}: {v:.4f}')

## 3. Per-axis mutation effects

`single_mutation_effects(ls, position=...)` returns one row per **(from-value, to-value)** swap on that axis, with the median absolute effect and the signed mean effect across all backgrounds where the swap is observed.

Because the three axes have different kinds, the *kind* of one-step move differs:

- `electrode` swap ⇒ change to any of the other 3 electrode materials (6 from-to pairs).
- `pH` / `potential_V` swap ⇒ change to any other grid level (3 pairs each).

Comparing the `mean_effect` magnitudes tells you which knob actually swings selectivity.

In [ ]:
for axis in ['electrode', 'pH', 'potential_V']:
    sm = single_mutation_effects(ls, position=axis)
    print(f'\n=== {axis} ({len(sm)} swap rows) ===')
    print(sm.sort_values('mean_effect', ascending=False).to_string(index=False))

## 4. Plot the fitness distribution

In [ ]:
from graphfla.plotting import draw_fitness_distribution
draw_fitness_distribution(ls)

## Notes

- Mixing axis types in `data_types` is the general workflow: declare each column individually.
- Add `'boolean'` for true binary inputs — equivalent to a 2-level categorical but skips the categorical-codes path for speed.
- If you flip the optimization direction (e.g. minimise gamma in `data/Pharmacology/Panc1/combinations.csv`), set `Landscape(maximize=False)` instead. Every analysis metric respects that direction.